# S5 · Schaefer100 spatial tests


In [1]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))


In [2]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
output_dir = f"{base_path}/FIGURES/supplementary/Fig_S5"
os.makedirs(output_dir, exist_ok=True)
data_path = f"{base_path}/RESULTS/replication_dataset/one_samp_per_session"

In [4]:
from utils.surface import SURFACE_VIEWS


# Plotting functions
from nilearn.surface import load_surf_mesh
from sklearn.neighbors import NearestNeighbors
import numpy as np
from nilearn import plotting, surface, datasets
import nibabel as nib
import matplotlib as mpl


def get_display_mesh(fsaverage, hemi, inflation=0.7):
    pial = surface.load_surf_mesh(getattr(fsaverage, f"pial_{hemi}"))
    infl = surface.load_surf_mesh(getattr(fsaverage, f"infl_{hemi}"))

    if not np.array_equal(pial.faces, infl.faces):
        raise ValueError("Surface meshes have different topology.")

    coords = (
        (1 - inflation) * pial.coordinates
        + inflation * infl.coordinates
    )

    return coords, pial.faces

def inpaint_surface_values_mean(values, mesh, n_neighbors=5):
    coords = load_surf_mesh(mesh)[0]  
    
    filled = values.copy()
    mask_nan = np.isnan(values)
    if not mask_nan.any():
        return filled
    
    mask_valid = ~mask_nan
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(coords[mask_valid])
    _, idx = nbrs.kneighbors(coords[mask_nan])
    
    neighbor_vals = values[mask_valid][idx]          
    filled[mask_nan] = np.nanmean(neighbor_vals, axis=1)
    return filled

def inpaint_surface_values(values, mesh):
    coords = load_surf_mesh(mesh)[0]  
    mask_nan = np.isnan(values)
    if not mask_nan.any():
        return values
    nbrs = NearestNeighbors(n_neighbors=1).fit(coords[~mask_nan])
    _, idx = nbrs.kneighbors(coords[mask_nan])
    filled = values.copy()
    filled[mask_nan] = values[~mask_nan][idx[:,0]]
    return filled

def plot_voxelwise(
    matrix,
    affine,
    img_name="voxelwise",
    cmap_custom="seismic",
    show=False,
    vmin=None,
    vmax=None,
    cbar_label="Value",
):

    LABEL_FS = 18
    TICK_FS = 18

    if vmin is None:
        vmin = np.nanmin(matrix)
    if vmax is None:
        vmax = np.nanmax(matrix)

    # Load fsaverage surface
    fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage6")

    views = list(SURFACE_VIEWS)

    fig, axes = plt.subplots(
        1,
        len(views),
        figsize=(10 * len(views), 10),
        subplot_kw={"projection": "3d"},
    )

    axes = np.array(axes).flatten()
    fig.subplots_adjust(left=0.02, right=0.9, wspace=0.00)

    # Convert voxelwise matrix to NIfTI image
    img = nib.Nifti1Image(matrix, affine=affine)

    # Project volume to cortical surface
    surf_map_lh = surface.vol_to_surf(
        img,
        fsaverage.pial_left,
        interpolation="linear",
        radius=5,
    )

    surf_map_rh = surface.vol_to_surf(
        img,
        fsaverage.pial_right,
        interpolation="linear",
        radius=5,
    )

    # Plot each hemisphere/view
    for j, (hemi, view) in enumerate(views):
        ax = axes[j]

        if hemi == "left":
            surf_mesh = get_display_mesh(fsaverage, hemi, inflation=0.7)
            surf_map = surf_map_lh
            bg_map = fsaverage.sulc_left
        else:
            surf_mesh = get_display_mesh(fsaverage, hemi, inflation=0.7)
            surf_map = surf_map_rh
            bg_map = fsaverage.sulc_right

        plotting.plot_surf_stat_map(
            surf_mesh,
            surf_map,
            hemi=hemi,
            view=view,
            cmap=cmap_custom,
            colorbar=False,
            bg_on_data=True,
            axes=ax,
            figure=fig,
            vmin=vmin,
            vmax=vmax,
            bg_map=bg_map,
            avg_method="median",
        )

    # Colorbar
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap_custom, norm=norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes,
        shrink=0.3,
        aspect=10,
        location="right",
        ticks=[vmin, vmax],
    )

    cbar.ax.tick_params(labelsize=TICK_FS)
    cbar.ax.set_ylabel(cbar_label, fontsize=LABEL_FS, fontweight="bold")

    fig.savefig(f"{img_name}.png", format="png", bbox_inches="tight", dpi=300)

    if show:
        plt.show()

    plt.close(fig)

In [5]:
threshold = 0.98
percentile_cut = 20  # top/bottom 10%

max_colors = ['#ff5e69', '#86ab00', '#03b6ce', '#8491ff'] 
tasks = ['rest', '0back', '1back', '2back']

for i in range(4):
    tstat_path = (
        f"{data_path}/OneSampT_voxelwise_residuals_divided_{tasks[i]}_tstat1.nii.gz"
    )
    pval_path = (
        f"{data_path}/OneSampT_voxelwise_residuals_divided_{tasks[i]}_tfce_corrp_tstat1.nii.gz"
    )

    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        f"{tasks[i]}_black_white_color",
        ["black", "white", max_colors[i]],
        N=256,
    )

    tstat_img = nib.load(tstat_path)
    data = tstat_img.get_fdata()

    pval_img = nib.load(pval_path)
    pval_data = pval_img.get_fdata()
    pval_mask = pval_data > threshold

    data[~pval_mask] = np.nan
    data[data == 0] = np.nan

    valid = data[~np.isnan(data)]
    p_low  = np.percentile(valid, percentile_cut)      
    p_high = np.percentile(valid, 100 - percentile_cut) 

    remapped = np.full_like(data, np.nan)
    remapped[(data >= p_low)  & (data < p_high)] = 0.0   
    remapped[data < p_low]  = -1.0                      
    remapped[data >= p_high] =  1.0                    

    print(f"{tasks[i]} | p_low={p_low:.2f}, p_high={p_high:.2f}")

    plot_voxelwise(
        remapped,
        tstat_img.affine,
        img_name=f"{output_dir}/voxelwise_tstat_{tasks[i]}_percentiles_30",
        cmap_custom=cmap,
        show=False,
        vmin=-1,
        vmax=1,
    )

rest | p_low=28.58, p_high=39.54
[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


0back | p_low=23.82, p_high=34.00
[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


1back | p_low=24.33, p_high=32.31
[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


2back | p_low=22.52, p_high=31.35
[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


In [6]:
threshold = 0.98

max_colors = ['#ff5e69', '#86ab00', '#03b6ce', '#8491ff'] 
tasks = ['rest', '0back', '1back', '2back']

for i in range(4):
    tstat_path = (
        f"{data_path}/OneSampT_voxelwise_residuals_divided_{tasks[i]}_tstat1.nii.gz"
    )
    pval_path = (
        f"{data_path}/OneSampT_voxelwise_residuals_divided_{tasks[i]}_tfce_corrp_tstat1.nii.gz"
    )

    cmap = mpl.colors.LinearSegmentedColormap.from_list(
        f"{tasks[i]}_white_color",
        ["white", max_colors[i]],
        N=256,
    )

    tstat_img = nib.load(tstat_path)
    data = tstat_img.get_fdata()

    pval_img = nib.load(pval_path)
    pval_data = pval_img.get_fdata()
    pval_mask = pval_data > threshold

    data[~pval_mask] = np.nan
    data[data == 0] = np.nan               

    plot_voxelwise(
        data,
        tstat_img.affine,
        img_name=f"{output_dir}/voxelwise_tstat_{tasks[i]}",
        cmap_custom=cmap,
        show=False,
        vmin=13,
        vmax=65,
    )

[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)


[fetch_surf_fsaverage] Dataset directory found: /home/matilde/nilearn_data/fsaverage6


/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
/home/matilde/miniconda3/envs/hurst-neuroimaging/lib/python3.11/site-packages/nilearn/surface/surface.py:1242: RuntimeWarning: Mean of empty slice
  texture = np.nanmean(all_samples, axis=2)
